In [1]:
import heapq

GOAL_STATE = (
    1, 2, 3, 4,
    5, 6, 7, 8,
    9, 10, 11, 12,
    13, 14, 15, 0
)

GOAL_POS = {value: (i // 4, i % 4) for i, value in enumerate(GOAL_STATE)}


class Node:
    def __init__(self, state, parent=None, move=None, g=0):
        self.state = state
        self.parent = parent
        self.move = move
        self.g = g
        self.h = self.heuristic()
        self.f = self.g + self.h

    def heuristic(self):
        # Manhattan distance
        dist = 0
        for i, value in enumerate(self.state):
            if value != 0:
                x1, y1 = i // 4, i % 4
                x2, y2 = GOAL_POS[value]
                dist += abs(x1 - x2) + abs(y1 - y2)
        return dist

    def __lt__(self, other):
        return self.f < other.f


def print_board(state):
    for i in range(0, 16, 4):
        row = state[i:i+4]
        print(" ".join(f"{x:2}" if x != 0 else " _" for x in row))
    print()


def get_neighbors(state):
    neighbors = []
    zero_index = state.index(0)
    x, y = zero_index // 4, zero_index % 4

    directions = {
        "Up": (-1, 0),
        "Down": (1, 0),
        "Left": (0, -1),
        "Right": (0, 1)
    }

    for move, (dx, dy) in directions.items():
        nx, ny = x + dx, y + dy
        if 0 <= nx < 4 and 0 <= ny < 4:
            new_index = nx * 4 + ny
            new_state = list(state)
            new_state[zero_index], new_state[new_index] = new_state[new_index], new_state[zero_index]
            neighbors.append((tuple(new_state), move))
    return neighbors


def reconstruct_path(node):
    path = []
    while node:
        path.append((node.state, node.move, node.g, node.h, node.f))
        node = node.parent
    return path[::-1]


def is_solvable(state):
    inv_count = 0
    arr = [x for x in state if x != 0]

    for i in range(len(arr)):
        for j in range(i + 1, len(arr)):
            if arr[i] > arr[j]:
                inv_count += 1

    zero_row_from_bottom = 4 - (state.index(0) // 4)

    # Với bảng 4x4:
    # - blank ở hàng chẵn từ dưới lên => nghịch thế phải lẻ
    # - blank ở hàng lẻ từ dưới lên => nghịch thế phải chẵn
    if zero_row_from_bottom % 2 == 0:
        return inv_count % 2 == 1
    else:
        return inv_count % 2 == 0


def akt_15_puzzle(start_state):
    if start_state == GOAL_STATE:
        return reconstruct_path(Node(start_state))

    if not is_solvable(start_state):
        return None

    open_list = []
    closed_set = set()
    g_best = {}

    start_node = Node(start_state)
    heapq.heappush(open_list, start_node)
    g_best[start_state] = 0

    while open_list:
        current = heapq.heappop(open_list)

        if current.state == GOAL_STATE:
            return reconstruct_path(current)

        if current.state in closed_set:
            continue
        closed_set.add(current.state)

        for next_state, move in get_neighbors(current.state):
            new_g = current.g + 1

            if next_state in closed_set:
                continue

            if next_state not in g_best or new_g < g_best[next_state]:
                g_best[next_state] = new_g
                child = Node(next_state, parent=current, move=move, g=new_g)
                heapq.heappush(open_list, child)

    return None


if __name__ == "__main__":
    start_state = (
        1, 2, 3, 4,
        5, 6, 0, 8,
        9, 10, 7, 12,
        13, 14, 11, 15
    )

    print("Trang thai ban dau:")
    print_board(start_state)

    result = akt_15_puzzle(start_state)

    if result is None:
        print("Khong tim thay loi giai hoac trang thai khong giai duoc.")
    else:
        print(f"Tim thay loi giai sau {len(result) - 1} buoc.\n")
        for i, (state, move, g, h, f) in enumerate(result):
            print(f"Buoc {i}: Move = {move}, g = {g}, h = {h}, f = {f}")
            print_board(state)

Trang thai ban dau:
 1  2  3  4
 5  6  _  8
 9 10  7 12
13 14 11 15

Tim thay loi giai sau 3 buoc.

Buoc 0: Move = None, g = 0, h = 3, f = 3
 1  2  3  4
 5  6  _  8
 9 10  7 12
13 14 11 15

Buoc 1: Move = Down, g = 1, h = 2, f = 3
 1  2  3  4
 5  6  7  8
 9 10  _ 12
13 14 11 15

Buoc 2: Move = Down, g = 2, h = 1, f = 3
 1  2  3  4
 5  6  7  8
 9 10 11 12
13 14  _ 15

Buoc 3: Move = Right, g = 3, h = 0, f = 3
 1  2  3  4
 5  6  7  8
 9 10 11 12
13 14 15  _



In [2]:
import heapq
import math


class Node:
    def __init__(self, state, size, parent=None, move=None, g=0):
        self.state = state
        self.size = size
        self.parent = parent
        self.move = move
        self.g = g
        self.h = self.heuristic()
        self.f = self.g + self.h

    def heuristic(self):
        # Manhattan Distance
        dist = 0
        for i, value in enumerate(self.state):
            if value != 0:
                goal_x = (value - 1) // self.size
                goal_y = (value - 1) % self.size
                cur_x = i // self.size
                cur_y = i % self.size
                dist += abs(cur_x - goal_x) + abs(cur_y - goal_y)
        return dist

    def __lt__(self, other):
        return self.f < other.f


def create_goal_state(size):
    return tuple(list(range(1, size * size)) + [0])


def print_board(state, size):
    width = len(str(size * size - 1))
    for i in range(size):
        row = state[i * size:(i + 1) * size]
        print(" ".join(f"{x:>{width}}" if x != 0 else "_" * width for x in row))
    print()


def get_neighbors(state, size):
    neighbors = []
    zero_index = state.index(0)
    x, y = zero_index // size, zero_index % size

    directions = {
        "Up": (-1, 0),
        "Down": (1, 0),
        "Left": (0, -1),
        "Right": (0, 1)
    }

    for move, (dx, dy) in directions.items():
        nx, ny = x + dx, y + dy
        if 0 <= nx < size and 0 <= ny < size:
            new_index = nx * size + ny
            new_state = list(state)
            new_state[zero_index], new_state[new_index] = new_state[new_index], new_state[zero_index]
            neighbors.append((tuple(new_state), move))
    return neighbors


def reconstruct_path(node):
    path = []
    while node:
        path.append((node.state, node.move, node.g, node.h, node.f))
        node = node.parent
    return path[::-1]


def count_inversions(state):
    arr = [x for x in state if x != 0]
    inv = 0
    for i in range(len(arr)):
        for j in range(i + 1, len(arr)):
            if arr[i] > arr[j]:
                inv += 1
    return inv


def is_solvable(state, size):
    inv = count_inversions(state)
    zero_row = state.index(0) // size
    zero_row_from_bottom = size - zero_row

    # Nếu size lẻ: số nghịch thế phải chẵn
    if size % 2 == 1:
        return inv % 2 == 0
    else:
        # Nếu size chẵn:
        # - ô trống ở hàng chẵn từ dưới lên => nghịch thế lẻ
        # - ô trống ở hàng lẻ từ dưới lên => nghịch thế chẵn
        if zero_row_from_bottom % 2 == 0:
            return inv % 2 == 1
        else:
            return inv % 2 == 0


def akt_n_puzzle(start_state, size, max_nodes=200000):
    goal_state = create_goal_state(size)

    if len(start_state) != size * size:
        raise ValueError("Kích thước trạng thái không đúng với size.")

    if set(start_state) != set(range(size * size)):
        raise ValueError("Trạng thái phải chứa đủ các số từ 0 đến size*size-1.")

    if start_state == goal_state:
        return reconstruct_path(Node(start_state, size))

    if not is_solvable(start_state, size):
        return None

    open_list = []
    closed_set = set()
    g_best = {}

    start_node = Node(start_state, size)
    heapq.heappush(open_list, start_node)
    g_best[start_state] = 0

    explored_nodes = 0

    while open_list:
        current = heapq.heappop(open_list)

        if current.state in closed_set:
            continue

        closed_set.add(current.state)
        explored_nodes += 1

        if explored_nodes > max_nodes:
            print("Vượt quá giới hạn số node cho phép, dừng tìm kiếm.")
            return None

        if current.state == goal_state:
            return reconstruct_path(current)

        for next_state, move in get_neighbors(current.state, size):
            if next_state in closed_set:
                continue

            new_g = current.g + 1

            if next_state not in g_best or new_g < g_best[next_state]:
                g_best[next_state] = new_g
                child = Node(
                    state=next_state,
                    size=size,
                    parent=current,
                    move=move,
                    g=new_g
                )
                heapq.heappush(open_list, child)

    return None


if __name__ == "__main__":
    size = 5   # ví dụ 5x5 => 24-puzzle

    start_state = (
        1, 2, 3, 4, 5,
        6, 7, 8, 9, 10,
        11, 12, 13, 14, 15,
        16, 17, 18, 0, 20,
        21, 22, 23, 19, 24
    )

    print("Trạng thái ban đầu:")
    print_board(start_state, size)

    result = akt_n_puzzle(start_state, size, max_nodes=500000)

    if result is None:
        print("Không tìm thấy lời giải, trạng thái không giải được, hoặc vượt giới hạn tìm kiếm.")
    else:
        print(f"Tìm thấy lời giải sau {len(result) - 1} bước.\n")
        for i, (state, move, g, h, f) in enumerate(result):
            print(f"Bước {i}: Move = {move}, g = {g}, h = {h}, f = {f}")
            print_board(state, size)

Trạng thái ban đầu:
 1  2  3  4  5
 6  7  8  9 10
11 12 13 14 15
16 17 18 __ 20
21 22 23 19 24

Tìm thấy lời giải sau 2 bước.

Bước 0: Move = None, g = 0, h = 2, f = 2
 1  2  3  4  5
 6  7  8  9 10
11 12 13 14 15
16 17 18 __ 20
21 22 23 19 24

Bước 1: Move = Down, g = 1, h = 1, f = 2
 1  2  3  4  5
 6  7  8  9 10
11 12 13 14 15
16 17 18 19 20
21 22 23 __ 24

Bước 2: Move = Right, g = 2, h = 0, f = 2
 1  2  3  4  5
 6  7  8  9 10
11 12 13 14 15
16 17 18 19 20
21 22 23 24 __

